# 03 — Multi-Head Attention from Scratch

Single-head attention can only learn one type of relationship at a time. Multi-head attention runs **multiple attention operations in parallel**, each learning to focus on different patterns.

## Table of Contents
1. [Why Multiple Heads?](#1-why-multiple-heads)
2. [How Heads Split and Recombine](#2-how-heads-split-and-recombine)
3. [Building Multi-Head Attention](#3-building-multi-head-attention)
4. [Comparing Single-Head vs Multi-Head](#4-comparing-single-head-vs-multi-head)
5. [Visualising Per-Head Attention Patterns](#5-visualising-per-head-attention-patterns)
6. [Key Takeaways](#6-key-takeaways)

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import sys, os

sys.path.insert(0, os.path.abspath('.'))
from utils.visualisation import (
    plot_attention_heatmap,
    plot_multi_head_attention
)

torch.manual_seed(42)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Why Multiple Heads?

A single attention head can only focus on **one type of pattern** per layer. But language has many simultaneous relationships:

- **Syntactic**: subject → verb agreement ("The dogs **run**")
- **Semantic**: word meaning similarities ("robot" ↔ "machine")
- **Positional**: adjacent word patterns (bigrams, trigrams)
- **Referential**: pronoun → antecedent ("it" → "the box")

Multi-head attention solves this by giving the model **multiple independent "attention eyes"**, each free to learn a different pattern.

### The Key Idea

Instead of one attention operation with $d_{model}$-dimensional Q, K, V:
- Split into $h$ heads, each with $d_k = d_{model} / h$ dimensions
- Run $h$ independent attention operations in parallel
- Concatenate the results and project back to $d_{model}$

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \cdot W^O$$

where $\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$

**Total parameters are the same** — we just distribute them across heads instead of using one big attention matrix.

## 2. How Heads Split and Recombine

Let's trace the dimension changes step by step.

In [ ]:
# ============================================================
# Setup: dimensions for our example
# ============================================================
batch_size = 1
seq_len = 6
d_model = 16     # total embedding dimension
n_heads = 4      # number of attention heads
d_k = d_model // n_heads  # per-head dimension

print(f"d_model = {d_model}")
print(f"n_heads = {n_heads}")
print(f"d_k = d_model / n_heads = {d_model} / {n_heads} = {d_k}")
print(f"")
print(f"Each token's {d_model}-dim embedding is split into {n_heads} heads of {d_k} dims each.")

# Create sample input
x = torch.randn(batch_size, seq_len, d_model)  # (1, 6, 16)
print(f"\nInput shape: {x.shape}  — (batch, seq_len, d_model)")

In [ ]:
# ============================================================
# Step 1: Project to Q, K, V (full d_model width)
# ============================================================
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

Q = W_q(x)  # (1, 6, 16)
K = W_k(x)  # (1, 6, 16)
V = W_v(x)  # (1, 6, 16)

print(f"After linear projection:")
print(f"  Q shape: {Q.shape}  — (batch, seq, d_model)")
print(f"  K shape: {K.shape}")
print(f"  V shape: {V.shape}")

In [ ]:
# ============================================================
# Step 2: Split into multiple heads
# ============================================================
# Reshape: (batch, seq, d_model) → (batch, seq, n_heads, d_k)
# Then transpose: → (batch, n_heads, seq, d_k)
#
# This puts n_heads in dim 1, so PyTorch's batched matmul
# processes all heads in parallel.

def split_heads(tensor, n_heads):
    """(batch, seq, d_model) → (batch, n_heads, seq, d_k)"""
    batch, seq, d_model = tensor.shape
    d_k = d_model // n_heads
    # (batch, seq, n_heads, d_k)
    tensor = tensor.view(batch, seq, n_heads, d_k)
    # (batch, n_heads, seq, d_k)
    return tensor.transpose(1, 2)

Q_heads = split_heads(Q, n_heads)  # (1, 4, 6, 4)
K_heads = split_heads(K, n_heads)
V_heads = split_heads(V, n_heads)

print(f"After splitting into {n_heads} heads:")
print(f"  Q_heads shape: {Q_heads.shape}  — (batch, n_heads, seq, d_k)")
print(f"  K_heads shape: {K_heads.shape}")
print(f"  V_heads shape: {V_heads.shape}")

print("\n" + "="*70)
print("VISUAL: How a single token's embedding is split")
print("="*70)
token_0_full = Q[0, 0].detach()  # 16-dim vector for token 0
print(f"\nToken 0 full Q vector ({d_model} dims): {token_0_full.numpy().round(3)}")
print(f"")
for h in range(n_heads):
    head_slice = Q_heads[0, h, 0].detach()  # token 0, head h
    start = h * d_k
    end = start + d_k
    print(f"  Head {h} gets dims [{start}:{end}]: {head_slice.numpy().round(3)}")

In [ ]:
# ============================================================
# Step 3: Attention per head (all in parallel via batched matmul)
# ============================================================
# scores: (batch, n_heads, seq, seq)
scores = torch.matmul(Q_heads, K_heads.transpose(-2, -1)) / math.sqrt(d_k)

# Causal mask
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
# Expand to (1, 1, seq, seq) for broadcasting over batch and heads
causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

scores = scores.masked_fill(causal_mask, float('-inf'))
attn_weights = F.softmax(scores, dim=-1)
attn_weights = torch.nan_to_num(attn_weights, nan=0.0)

# attn_output: (batch, n_heads, seq, d_k)
attn_output = torch.matmul(attn_weights, V_heads)

print(f"Attention scores shape:  {scores.shape}  — (batch, heads, seq, seq)")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"Attention output shape:  {attn_output.shape}  — (batch, heads, seq, d_k)")

print("\n" + "-"*60)
print("Each head has its OWN (seq × seq) attention matrix.")
print(f"We have {n_heads} heads → {n_heads} different attention patterns.")
print("-"*60)

In [ ]:
# ============================================================
# Step 4: Merge heads back together
# ============================================================
# (batch, n_heads, seq, d_k) → (batch, seq, n_heads, d_k) → (batch, seq, d_model)

def merge_heads(tensor):
    """(batch, n_heads, seq, d_k) → (batch, seq, d_model)"""
    batch, n_heads, seq, d_k = tensor.shape
    # (batch, seq, n_heads, d_k)
    tensor = tensor.transpose(1, 2).contiguous()
    # (batch, seq, d_model)  — d_model = n_heads × d_k
    return tensor.view(batch, seq, n_heads * d_k)

merged = merge_heads(attn_output)  # (1, 6, 16)
print(f"Merged output shape: {merged.shape}  — back to (batch, seq, d_model)")

# Step 5: Final output projection
W_o = nn.Linear(d_model, d_model, bias=False)
final_output = W_o(merged)  # (1, 6, 16)
print(f"Final output shape:  {final_output.shape}")

In [ ]:
# ============================================================
# Complete data flow summary
# ============================================================

print("="*70)
print("COMPLETE DATA FLOW: Multi-Head Attention")
print("="*70)
print(f"""
Input:        (1, 6, 16)   ← {batch_size} batch, {seq_len} tokens, {d_model}D embeddings
    │
    ├─ W_q ──→ Q: (1, 6, 16)
    ├─ W_k ──→ K: (1, 6, 16)     ← Linear projections
    └─ W_v ──→ V: (1, 6, 16)
    │
Split heads:  (1, {n_heads}, 6, {d_k})    ← {n_heads} heads, each {d_k}D
    │
Attention:    (1, {n_heads}, 6, 6)     ← {n_heads} attention matrices
    │
Weighted V:   (1, {n_heads}, 6, {d_k})    ← attended values per head
    │
Merge heads:  (1, 6, 16)   ← concatenate: {n_heads}×{d_k}D = {d_model}D
    │
W_o:          (1, 6, 16)   ← output projection (mixes head outputs)
""")

## 3. Building Multi-Head Attention

Let's wrap this into a clean `nn.Module` class.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Self-Attention built from scratch.
    
    Args:
        d_model:  total embedding dimension
        n_heads:  number of parallel attention heads
        dropout:  dropout rate on attention weights
    
    Input:  (batch, seq, d_model)
    Output: (batch, seq, d_model)
    """
    
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # per-head dimension
        
        # Projection matrices (one big matrix for all heads combined)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        
        # Store attention weights for visualisation
        self.last_attn_weights = None
    
    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape
        
        # 1. Project to Q, K, V
        Q = self.W_q(x)  # (batch, seq, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # 2. Split into heads
        # (batch, seq, d_model) → (batch, n_heads, seq, d_k)
        Q = Q.view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)
        
        # 3. Scaled dot-product attention (all heads in parallel)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        
        # Save for visualisation
        self.last_attn_weights = attn_weights.detach()
        
        attn_weights = self.dropout(attn_weights)
        attn_output = torch.matmul(attn_weights, V)  # (batch, n_heads, seq, d_k)
        
        # 4. Merge heads
        # (batch, n_heads, seq, d_k) → (batch, seq, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch, seq, self.d_model)
        
        # 5. Output projection
        output = self.W_o(attn_output)
        
        return output

# Test it
mha = MultiHeadAttention(d_model=16, n_heads=4)
x_test = torch.randn(2, 6, 16)  # batch=2, seq=6, d=16

# Causal mask: (1, 1, seq, seq) for broadcasting
mask = torch.triu(torch.ones(6, 6, dtype=torch.bool), diagonal=1).unsqueeze(0).unsqueeze(0)

out = mha(x_test, mask=mask)
print(f"Input shape:  {x_test.shape}")
print(f"Output shape: {out.shape}")
print(f"Attention weights shape: {mha.last_attn_weights.shape}  — (batch, heads, seq, seq)")
print(f"\n✓ Multi-Head Attention module works correctly.")

## 4. Comparing Single-Head vs Multi-Head

Let's see the difference in attention patterns between 1 head and 4 heads, using the same input.

In [ ]:
# ============================================================
# Compare: 1 head vs 4 heads on the same input
# ============================================================

torch.manual_seed(42)

tokens = ["The", "robot", "picked", "up", "the", "box"]
x_compare = torch.randn(1, 6, 16)
mask_compare = torch.triu(torch.ones(6, 6, dtype=torch.bool), diagonal=1).unsqueeze(0).unsqueeze(0)

# Single head (d_model=16, n_heads=1 → d_k=16)
single_head = MultiHeadAttention(d_model=16, n_heads=1)
_ = single_head(x_compare, mask=mask_compare)
single_attn = single_head.last_attn_weights  # (1, 1, 6, 6)

# Multi-head (d_model=16, n_heads=4 → d_k=4)
multi_head = MultiHeadAttention(d_model=16, n_heads=4)
_ = multi_head(x_compare, mask=mask_compare)
multi_attn = multi_head.last_attn_weights  # (1, 4, 6, 6)

# Plot single head
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

# Single head
plot_attention_heatmap(
    single_attn[0, 0], tokens, title="Single Head (d_k=16)",
    ax=axes[0], annot=True, fmt=".2f", cbar=False
)

# Multi-head: each head
for h in range(4):
    plot_attention_heatmap(
        multi_attn[0, h], tokens, title=f"Head {h} (d_k=4)",
        ax=axes[h+1], annot=True, fmt=".2f", cbar=False
    )

plt.suptitle("Single Head vs Multi-Head Attention Patterns",
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("KEY INSIGHT:")
print("="*70)
print("With 1 head: only one attention pattern — one 'view' of the data.")
print("With 4 heads: four DIFFERENT patterns — each head can specialise.")
print("")
print("After training, different heads often learn interpretable roles:")
print("  • Head 0 might track adjacent tokens (local patterns)")
print("  • Head 1 might focus on the first token (positional)")
print("  • Head 2 might link syntactically related words")
print("  • Head 3 might capture semantic similarity")
print("="*70)

## 5. Visualising Per-Head Attention Patterns

Let's use the `plot_multi_head_attention` helper for a cleaner grid view.

In [ ]:
# ============================================================
# Use the shared visualisation helper
# ============================================================

plot_multi_head_attention(
    multi_attn[0],  # (4, 6, 6) — all heads for batch 0
    tokens=tokens,
    title="Per-Head Attention Patterns (4 Heads, Causal Masked)",
    annot=True
)

In [ ]:
# ============================================================
# Average attention across heads
# ============================================================
# This shows the "overall" attention pattern the model uses

avg_attn = multi_attn[0].mean(dim=0)  # (6, 6) — average over 4 heads

plot_attention_heatmap(
    avg_attn,
    tokens=tokens,
    title="Average Attention Across All Heads",
    figsize=(7, 6)
)

print("The average combines all heads' perspectives into one view.")
print("This is useful for understanding overall information flow,")
print("but individual heads reveal more specific patterns.")

In [ ]:
# ============================================================
# Measure head diversity: how different are the attention patterns?
# ============================================================

print("="*60)
print("HEAD DIVERSITY ANALYSIS")
print("="*60)

# Pairwise cosine similarity between attention patterns
# Flatten each head's attention matrix to a vector, then compare
head_vectors = multi_attn[0].reshape(n_heads, -1)  # (4, 36)

print(f"\nPairwise cosine similarity between heads:")
print(f"{'':>8}", end="")
for h in range(n_heads):
    print(f"  Head {h}", end="")
print()

for i in range(n_heads):
    print(f"Head {i}:", end="")
    for j in range(n_heads):
        sim = F.cosine_similarity(head_vectors[i:i+1], head_vectors[j:j+1]).item()
        print(f"  {sim:5.3f} ", end="")
    print()

print("\nLower off-diagonal values = more diverse heads (good!)")
print("High similarity between heads = redundancy (wasted capacity)")

## 6. Key Takeaways

---

### Summary

| Concept | Details |
|---------|--------|
| **Why multi-head?** | Different heads learn different relationship patterns |
| **How it works** | Split d_model into n_heads × d_k, run attention in parallel |
| **Parameter count** | Same as single-head (just redistributed) |
| **Concatenation** | After attention, head outputs are concatenated back to d_model |
| **Output projection (W_o)** | Lets heads' information mix and interact |
| **Diversity** | Ideally, each head specialises in a different pattern |

### What's Next?

In the next notebook (**04 — Transformer Block**), we'll combine multi-head attention with feed-forward networks, layer normalisation, and residual connections to build a complete transformer block.

---